In [1]:
# !rm -rf /kaggle/working/SwinLLIE
# !rm -rf /kaggle/working/Diffunet
# !rm -rf /kaggle/working/Zero_DCE-LowLight
# !rm -rf /kaggle/working/Diffusion_new_final

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q fastapi uvicorn python-multipart pyngrok pillow timm einops scikit-image nest_asyncio

In [ ]:
import os, sys, subprocess

os.makedirs("/kaggle/working", exist_ok=True)
os.chdir("/kaggle/working")

# ── SwinLLIE ──
REPO_URL = "https://github.com/kavindamihiran/SwinLLIE.git"
BRANCH   = "final"
if not os.path.exists("/kaggle/working/SwinLLIE"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, "SwinLLIE"], check=True)
sys.path.insert(0, "/kaggle/working/SwinLLIE")

# ── Brightness Classifier ──
CLASSIFIER_REPO   = "https://github.com/Rashmika-D-N/Bclassify"
CLASSIFIER_BRANCH = "main"
CLASSIFIER_DIR    = "/kaggle/working/Bclassify"
if not os.path.exists(CLASSIFIER_DIR):
    subprocess.run(["git", "clone", "-b", CLASSIFIER_BRANCH, CLASSIFIER_REPO, CLASSIFIER_DIR], check=True)
sys.path.insert(0, CLASSIFIER_DIR)

# ── DiffuNet (NEW) ──
DIFFUNET_REPO   = "https://github.com/chirana07/Diffusion_new_final.git"
DIFFUNET_BRANCH = "main"
DIFFUNET_DIR    = "/kaggle/working/Diffusion_new_final"
if not os.path.exists(DIFFUNET_DIR):
    subprocess.run(["git", "clone", "-b", DIFFUNET_BRANCH, DIFFUNET_REPO, DIFFUNET_DIR], check=True)

os.chdir("/kaggle/working/SwinLLIE")
print("All repos ready (SwinLLIE, Classifier, Zero-DCE, DiffuNet)")

Cloning into 'SwinLLIE'...
Updating files: 100% (1380/1380), done.
Filtering content: 100% (2/2), 78.31 MiB | 16.75 MiB/s, done.
Cloning into '/kaggle/working/Bclassify'...
Cloning into '/kaggle/working/Diffusion_new_final'...


All repos ready (SwinLLIE, Classifier, Zero-DCE, DiffuNet)


In [4]:
import torch
from swinllie import SwinLLIE

CHECKPOINT_PATH = "./experiments/test_run/checkpoints/best.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

quality_model = SwinLLIE()
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
quality_model.load_state_dict(ckpt["model_state_dict"], strict=False)
quality_model = quality_model.to(DEVICE).eval()
print("✅ SwinLLIE (quality) loaded!")

Using device: cuda
✅ SwinLLIE (quality) loaded!


In [5]:
import os

# List everything under /kaggle/input so we can find the .pth file
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".pth"):
            print(os.path.join(root, f))

/kaggle/input/datasets/kavindamihiran/classifier/model.pth
/kaggle/input/datasets/kavindamihiran/zerodce/Epoch99.pth


In [ ]:
import importlib.util

# ────────────────────────────────────────────
# Zero-DCE shared loader (UNCHANGED)
# ────────────────────────────────────────────
ZDCE_REPO   = "https://github.com/Shampavi-Premananthan/Zero_DCE-LowLight.git"
ZDCE_BRANCH = "kaggle"
ZDCE_DIR    = "/kaggle/working/Zero_DCE-LowLight"
ZDCE_CODE   = f"{ZDCE_DIR}/Zero-DCE/Zero-DCE_code"

if not os.path.exists(ZDCE_DIR):
    subprocess.run(["git", "clone", "-b", ZDCE_BRANCH, ZDCE_REPO, ZDCE_DIR], check=True)

spec = importlib.util.spec_from_file_location("zdce_model", f"{ZDCE_CODE}/model.py")
zdce_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(zdce_module)
enhance_net_nopool = zdce_module.enhance_net_nopool

# ── Zero-DCE Model 1 — "dark" (UNCHANGED) ──
ZDCE_CKPT_DARK = "/kaggle/input/datasets/kavindamihiran/zerodce/Epoch99.pth"

zdce_model_dark = enhance_net_nopool()
ckpt1 = torch.load(ZDCE_CKPT_DARK, map_location=DEVICE)
md1 = zdce_model_dark.state_dict()
md1.update({k: v for k, v in ckpt1.items() if k in md1})
zdce_model_dark.load_state_dict(md1)
zdce_model_dark = zdce_model_dark.to(DEVICE).eval()
print("✅ Zero-DCE Model 1 (dark) loaded!")

# ── Zero-DCE Model 2 — "very_dark" (UNCHANGED) ──
ZDCE_CKPT_VERY_DARK = "/kaggle/input/datasets/kavindamihiran/zerodce/Epoch99.pth"

zdce_model_very_dark = enhance_net_nopool()
ckpt2 = torch.load(ZDCE_CKPT_VERY_DARK, map_location=DEVICE)
md2 = zdce_model_very_dark.state_dict()
md2.update({k: v for k, v in ckpt2.items() if k in md2})
zdce_model_very_dark.load_state_dict(md2)
zdce_model_very_dark = zdce_model_very_dark.to(DEVICE).eval()
print("✅ Zero-DCE Model 2 (very_dark) loaded!")

# ── Brightness Classifier (UNCHANGED) ──
from torchvision import transforms
from PIL import ImageStat

classifier_spec = importlib.util.spec_from_file_location(
    "classifier_model", f"{CLASSIFIER_DIR}/model.py"
)
classifier_module = importlib.util.module_from_spec(classifier_spec)
classifier_spec.loader.exec_module(classifier_module)

CLASSIFIER_CKPT = f"{CLASSIFIER_DIR}/model.pth"

classifier_model = classifier_module.get_model(num_classes=2)
if os.path.exists(CLASSIFIER_CKPT):
    classifier_model.load_state_dict(torch.load(CLASSIFIER_CKPT, map_location=DEVICE))
    print("✅ Brightness classifier loaded!")
else:
    print("⚠️  classifier model.pth not found — brightness fallback only")

classifier_model = classifier_model.to(DEVICE).eval()

classifier_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

CLASSIFIER_CLASSES = ['dark', 'very_dark']

# ════════════════════════════════════════════════════════
# DiffuNet — NEW
# ════════════════════════════════════════════════════════
sys.path.insert(0, DIFFUNET_DIR)

# Load config.py
diffunet_config_spec = importlib.util.spec_from_file_location(
    "diffunet_config", f"{DIFFUNET_DIR}/config.py"
)
diffunet_config_module = importlib.util.module_from_spec(diffunet_config_spec)
diffunet_config_spec.loader.exec_module(diffunet_config_module)

# Load model.py
diffunet_model_spec = importlib.util.spec_from_file_location(
    "diffunet_model", f"{DIFFUNET_DIR}/model.py"
)
diffunet_model_module = importlib.util.module_from_spec(diffunet_model_spec)
diffunet_model_spec.loader.exec_module(diffunet_model_module)

# Load diffusion.py
diffunet_diffusion_spec = importlib.util.spec_from_file_location(
    "diffunet_diffusion", f"{DIFFUNET_DIR}/diffusion.py"
)
diffunet_diffusion_module = importlib.util.module_from_spec(diffunet_diffusion_spec)
diffunet_diffusion_spec.loader.exec_module(diffunet_diffusion_module)

# Instantiate
DIFFUNET_CKPT = "/kaggle/working/Diffusion_new_final/checkpoints/last_pth_only/final.pth"  # ← UPDATE

diffunet_model = diffunet_model_module.ResidualConditionedUNet().to(DEVICE)
diffunet_engine = diffunet_diffusion_module.DiffusionEngine()

diffunet_ckpt = torch.load(DIFFUNET_CKPT, map_location=DEVICE)
if "ema" in diffunet_ckpt:
    diffunet_model.load_state_dict(diffunet_ckpt["ema"])
    print("✅ DiffuNet loaded (EMA weights)!")
elif "model" in diffunet_ckpt:
    diffunet_model.load_state_dict(diffunet_ckpt["model"])
    print("✅ DiffuNet loaded (standard weights)!")
else:
    diffunet_model.load_state_dict(diffunet_ckpt)
    print("✅ DiffuNet loaded (raw state_dict)!")

diffunet_model.eval()

os.chdir("/kaggle/working/SwinLLIE")
print("✅ ALL models loaded! (Zero-DCE×2, Classifier, SwinLLIE, DiffuNet)")

Cloning into '/kaggle/working/Zero_DCE-LowLight'...


✅ Zero-DCE Model 1 (dark) loaded!
✅ Zero-DCE Model 2 (very_dark) loaded!
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 121MB/s]


✅ Brightness classifier loaded!
✅ DiffuNet loaded (EMA weights)!
✅ ALL models loaded! (Zero-DCE×2, Classifier, SwinLLIE, DiffuNet)


In [ ]:
import io, uuid, time, base64
import numpy as np
from PIL import Image
import torch
import torchvision.transforms.functional as TF

sys.path.insert(0, "/kaggle/working/SwinLLIE")
from swinllie.utils import calculate_psnr, calculate_ssim
print("✅ Using SwinLLIE's calculate_psnr & calculate_ssim")

# ── Shared helpers (UNCHANGED) ─────────────────────────
def tensor_to_pil(t):
    arr = t.squeeze(0).permute(1, 2, 0).cpu().numpy()
    arr = np.clip(arr * 255, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def pil_to_tensor(img):
    arr = np.array(img).astype(np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)

def pad_to_multiple(x, unit=32):
    H, W = x.shape[2], x.shape[3]
    pad_h = (unit - H % unit) % unit
    pad_w = (unit - W % unit) % unit
    return torch.nn.functional.pad(x, (0, pad_w, 0, pad_h), mode="reflect"), H, W

def compute_metrics(orig_pil, enhanced_pil):
    orig_arr     = np.array(orig_pil)
    enhanced_arr = np.array(enhanced_pil)
    assert orig_arr.shape == enhanced_arr.shape
    p = round(float(calculate_psnr(orig_arr, enhanced_arr)), 2)
    s = round(float(calculate_ssim(orig_arr, enhanced_arr)), 4)
    return p, s

# ── Tile inference for SwinLLIE (UNCHANGED) ─────────────
TILE_SIZE    = 512
TILE_OVERLAP = 32

def tile_inference(x, mdl, tile=TILE_SIZE, overlap=TILE_OVERLAP):
    B, C, H, W = x.shape
    stride = tile - overlap
    def get_starts(dim):
        if dim <= tile:
            return [0]
        starts = list(range(0, dim - tile, stride))
        if starts[-1] + tile < dim:
            starts.append(dim - tile)
        return starts
    h_starts = get_starts(H)
    w_starts = get_starts(W)
    output = torch.zeros(B, C, H, W, dtype=x.dtype, device=x.device)
    weight = torch.zeros(B, 1, H, W, dtype=x.dtype, device=x.device)
    with torch.no_grad():
        for hs in h_starts:
            he = min(hs + tile, H)
            for ws in w_starts:
                we = min(ws + tile, W)
                tile_in  = x[:, :, hs:he, ws:we]
                tile_out = mdl(tile_in)
                output[:, :, hs:he, ws:we] += tile_out
                weight[:, :, hs:he, ws:we] += 1.0
                if x.device.type == "cuda":
                    torch.cuda.empty_cache()
    return output / weight.clamp(min=1e-8)

# ── Classifier (UNCHANGED) ─────────────────────────────
def classify_brightness(img: Image.Image) -> str:
    grayscale = img.convert('L')
    stat = ImageStat.Stat(grayscale)
    avg_brightness = stat.mean[0]
    if avg_brightness > 80:
        return 'dark'
    input_tensor = classifier_transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        outputs = classifier_model(input_tensor)
        _, predicted = outputs.max(1)
        category = CLASSIFIER_CLASSES[predicted.item()]
    return category

# ── Zero-DCE runner (UNCHANGED) ────────────────────────
def _run_zdce(img_bytes: bytes, zdce_model, iterations: int):
    start = time.time()
    orig = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    x = pil_to_tensor(orig).to(DEVICE)
    padded, H, W = pad_to_multiple(x, unit=32)
    
    with torch.no_grad():
        out = padded
        # --- ITERATION LOOP ---
        for _ in range(iterations):
            model_out = zdce_model(out)
            out = model_out[0] if isinstance(model_out, (tuple, list)) else model_out
            
    out = out[:, :, :H, :W]
    enhanced = tensor_to_pil(out)
    p, s = compute_metrics(orig, enhanced)
    buf = io.BytesIO(); enhanced.save(buf, format="PNG")
    elapsed = round(time.time() - start, 2)
    return buf.getvalue(), p, s, elapsed

# ── enhance_fast (UNCHANGED) ──────────────────────────
def enhance_fast(img_bytes: bytes):
    orig = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    category = classify_brightness(orig)
    print(f"🔍 Classified as: {category}")
    
    if category == "very_dark":
        # VERY LOW LIGHT: Run 8 iterations
        result = _run_zdce(img_bytes, zdce_model_very_dark, iterations=2)
        return result + (category,)
    else:
        # NORMAL/DARK LIGHT: Run 4 iterations
        result = _run_zdce(img_bytes, zdce_model_dark, iterations=1)
        return result + (category,)

# ── enhance_quality / SwinLLIE (UNCHANGED) ─────────────
def enhance_quality(img_bytes: bytes):
    start = time.time()
    orig = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    x = pil_to_tensor(orig).to(DEVICE)
    out = tile_inference(x, quality_model, tile=TILE_SIZE, overlap=TILE_OVERLAP)
    enhanced = tensor_to_pil(out)
    p, s = compute_metrics(orig, enhanced)
    buf = io.BytesIO(); enhanced.save(buf, format="PNG")
    elapsed = round(time.time() - start, 2)
    return buf.getvalue(), p, s, elapsed

# ════════════════════════════════════════════════════════
# DiffuNet enhancement — TILED LARGE-IMAGE VERSION
# ════════════════════════════════════════════════════════

DIFFUNET_TILE = 192        # use 128 if still OOM, 256 if stable
DIFFUNET_OVERLAP = 48
DIFFUNET_STEPS = 5

def _pad_tensor_to_multiple_of_8(x: torch.Tensor):
    _, _, h, w = x.shape
    pad_h = (8 - h % 8) % 8
    pad_w = (8 - w % 8) % 8
    x = torch.nn.functional.pad(x, (0, pad_w, 0, pad_h), mode="reflect")
    return x, h, w

def _get_tile_starts(length: int, tile: int, overlap: int):
    if length <= tile:
        return [0]
    stride = tile - overlap
    starts = list(range(0, length - tile + 1, stride))
    if starts[-1] != length - tile:
        starts.append(length - tile)
    return starts

def _make_blend_mask(h: int, w: int, device, dtype):
    wy = torch.hann_window(h, periodic=False, device=device, dtype=dtype).clamp_min(1e-3)
    wx = torch.hann_window(w, periodic=False, device=device, dtype=dtype).clamp_min(1e-3)
    mask = wy[:, None] * wx[None, :]
    return mask.unsqueeze(0).unsqueeze(0)

def diffunet_tile_sample(low_tensor: torch.Tensor,
                         tile: int = DIFFUNET_TILE,
                         overlap: int = DIFFUNET_OVERLAP,
                         inference_steps: int = DIFFUNET_STEPS):
    low_tensor, orig_h, orig_w = _pad_tensor_to_multiple_of_8(low_tensor)
    b, c, H, W = low_tensor.shape

    if H <= tile and W <= tile:
        with torch.inference_mode():
            out = diffunet_engine.ddim_sample(
                diffunet_model,
                low_tensor,
                inference_steps=inference_steps,
                eta=0.0,
            )
        return out[:, :, :orig_h, :orig_w]

    h_starts = _get_tile_starts(H, tile, overlap)
    w_starts = _get_tile_starts(W, tile, overlap)

    output = torch.zeros_like(low_tensor)
    weight = torch.zeros((b, 1, H, W), device=low_tensor.device, dtype=low_tensor.dtype)

    with torch.inference_mode():
        for hs in h_starts:
            he = hs + tile
            for ws in w_starts:
                we = ws + tile

                tile_in = low_tensor[:, :, hs:he, ws:we]

                tile_out = diffunet_engine.ddim_sample(
                    diffunet_model,
                    tile_in,
                    inference_steps=inference_steps,
                    eta=0.0,
                )

                blend = _make_blend_mask(
                    tile_out.shape[-2],
                    tile_out.shape[-1],
                    tile_out.device,
                    tile_out.dtype
                )

                output[:, :, hs:he, ws:we] += tile_out * blend
                weight[:, :, hs:he, ws:we] += blend

                del tile_in, tile_out, blend
                if DEVICE == "cuda":
                    torch.cuda.empty_cache()

    output = output / weight.clamp_min(1e-6)
    return output[:, :, :orig_h, :orig_w]

def enhance_diffunet(img_bytes: bytes):
    start = time.time()
    orig = Image.open(io.BytesIO(img_bytes)).convert("RGB")

    low_tensor = TF.to_tensor(orig).unsqueeze(0).to(DEVICE)
    low_tensor = (low_tensor - 0.5) * 2.0

    gen_tensor = diffunet_tile_sample(
        low_tensor,
        tile=DIFFUNET_TILE,
        overlap=DIFFUNET_OVERLAP,
        inference_steps=DIFFUNET_STEPS
    )

    gen_tensor = (gen_tensor + 1.0) / 2.0
    gen_tensor = torch.clamp(gen_tensor, 0.0, 1.0)

    enhanced = TF.to_pil_image(gen_tensor.squeeze(0).cpu())

    p, s = compute_metrics(orig, enhanced)

    buf = io.BytesIO()
    enhanced.save(buf, format="PNG")
    elapsed = round(time.time() - start, 2)

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return buf.getvalue(), p, s, elapsed

print("✅ All pipelines ready (classifier + dual Zero-DCE + SwinLLIE + DiffuNet)")

✅ Using SwinLLIE's calculate_psnr & calculate_ssim
✅ All pipelines ready (classifier + dual Zero-DCE + SwinLLIE + DiffuNet)


In [8]:
import uvicorn
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title="Low-Light Enhancement API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

SESSIONS: dict[str, bytes] = {}

class QualityRequest(BaseModel):
    session_id: str

# ──────────────────────────────────────────
# Health
# ──────────────────────────────────────────

@app.get("/")
def root():
    return {"status": "ok", "gpu": DEVICE}

@app.get("/health")
def health():
    return {
        "status": "ok",
        "gpu": DEVICE,
        "cuda_name": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "N/A",
        "models": {
            "fast": ["zerodce_dark", "zerodce_very_dark"],
            "quality": "swinllie",
            "diffunet": "diffunet",
            "classifier": "brightness_classifier",
        },
    }

# ──────────────────────────────────────────
# FAST — classifier picks Zero-DCE (UNCHANGED)
# ──────────────────────────────────────────

@app.post("/api/enhance/fast")
async def api_enhance_fast(image: UploadFile = File(...)):
    allowed = {"image/jpeg", "image/jpg", "image/png"}
    if image.content_type not in allowed:
        raise HTTPException(400, "Only JPG / PNG allowed.")
    contents = await image.read()
    if len(contents) > 10 * 1024 * 1024:
        raise HTTPException(400, "File too large (max 10 MB).")

    session_id = str(uuid.uuid4())
    SESSIONS[session_id] = contents

    try:
        enhanced_bytes, p, s, elapsed, category = enhance_fast(contents)
    except Exception as e:
        import traceback; traceback.print_exc()
        raise HTTPException(500, str(e))

    orig_b64     = base64.b64encode(contents).decode()
    enhanced_b64 = base64.b64encode(enhanced_bytes).decode()

    return {
        "success": True,
        "session_id": session_id,
        "original_url": f"data:image/png;base64,{orig_b64}",
        "image_url":    f"data:image/png;base64,{enhanced_b64}",
        "psnr": p, "ssim": s,
        "mode": "fast",
        "model_used": f"zerodce_{category}",
        "brightness_class": category,
        "processing_time": elapsed,
    }

# ──────────────────────────────────────────
# QUALITY — SwinLLIE (UNCHANGED)
# ──────────────────────────────────────────

@app.post("/api/enhance/quality")
async def api_enhance_quality(req: QualityRequest):
    contents = SESSIONS.get(req.session_id)
    if contents is None:
        raise HTTPException(404, "Session not found. Please upload image again.")
    try:
        enhanced_bytes, p, s, elapsed = enhance_quality(contents)
    except Exception as e:
        import traceback; traceback.print_exc()
        raise HTTPException(500, str(e))

    enhanced_b64 = base64.b64encode(enhanced_bytes).decode()
    return {
        "success": True,
        "session_id": req.session_id,
        "image_url": f"data:image/png;base64,{enhanced_b64}",
        "psnr": p, "ssim": s,
        "mode": "quality",
        "model": "swinllie",
        "processing_time": elapsed,
    }

# ──────────────────────────────────────────
# DIFFUNET — NEW high-quality endpoint
# ──────────────────────────────────────────

@app.post("/api/enhance/diffunet")
async def api_enhance_diffunet(req: QualityRequest):
    """
    DiffuNet enhancement using session_id (image already uploaded via /fast).
    Same contract as /api/enhance/quality — just a different model.
    """
    contents = SESSIONS.get(req.session_id)
    if contents is None:
        raise HTTPException(404, "Session not found. Please upload image again.")
    try:
        enhanced_bytes, p, s, elapsed = enhance_diffunet(contents)
    except Exception as e:
        import traceback; traceback.print_exc()
        raise HTTPException(500, str(e))

    enhanced_b64 = base64.b64encode(enhanced_bytes).decode()
    return {
        "success": True,
        "session_id": req.session_id,
        "image_url": f"data:image/png;base64,{enhanced_b64}",
        "psnr": p, "ssim": s,
        "mode": "quality",
        "model": "diffunet",
        "processing_time": elapsed,
    }


# ──────────────────────────────────────────
# ZDCE DARK / VERY DARK — Specific endpoints
# ──────────────────────────────────────────

@app.post("/api/enhance/zdce_dark")
async def api_enhance_zdce_dark(req: QualityRequest):
    contents = SESSIONS.get(req.session_id)
    if contents is None:
        raise HTTPException(404, "Session not found. Please upload image again.")
    try:
        # Note: _run_zdce returns (enhanced_bytes, p, s, elapsed, iterations)
        enhanced_bytes, p, s, elapsed = _run_zdce(contents, zdce_model_dark, iterations=1)
    except Exception as e:
        import traceback; traceback.print_exc()
        raise HTTPException(500, str(e))

    enhanced_b64 = base64.b64encode(enhanced_bytes).decode()
    return {
        "success": True,
        "session_id": req.session_id,
        "image_url": f"data:image/png;base64,{enhanced_b64}",
        "psnr": p, "ssim": s,
        "mode": "quality",
        "model": "zerodce_dark",
        "processing_time": elapsed,
    }

@app.post("/api/enhance/zdce_very_dark")
async def api_enhance_zdce_very_dark(req: QualityRequest):
    contents = SESSIONS.get(req.session_id)
    if contents is None:
        raise HTTPException(404, "Session not found. Please upload image again.")
    try:
        enhanced_bytes, p, s, elapsed = _run_zdce(contents, zdce_model_very_dark, iterations=2)
    except Exception as e:
        import traceback; traceback.print_exc()
        raise HTTPException(500, str(e))

    enhanced_b64 = base64.b64encode(enhanced_bytes).decode()
    return {
        "success": True,
        "session_id": req.session_id,
        "image_url": f"data:image/png;base64,{enhanced_b64}",
        "psnr": p, "ssim": s,
        "mode": "quality",
        "model": "zerodce_very_dark",
        "processing_time": elapsed,
    }

# ──────────────────────────────────────────
# Direct upload endpoints (optional)
# ──────────────────────────────────────────

@app.post("/api/enhance/zerodce")
async def api_enhance_zerodce(image: UploadFile = File(...)):
    allowed = {"image/jpeg", "image/jpg", "image/png"}
    if image.content_type not in allowed:
        raise HTTPException(400, "Only JPG / PNG allowed.")
    contents = await image.read()
    if len(contents) > 10 * 1024 * 1024:
        raise HTTPException(400, "File too large (max 10 MB).")

    session_id = str(uuid.uuid4())
    SESSIONS[session_id] = contents

    try:
        enhanced_bytes, p, s, elapsed, category = enhance_fast(contents)
    except Exception as e:
        raise HTTPException(500, str(e))

    orig_b64     = base64.b64encode(contents).decode()
    enhanced_b64 = base64.b64encode(enhanced_bytes).decode()

    return {
        "success": True,
        "session_id": session_id,
        "original_url": f"data:image/png;base64,{orig_b64}",
        "image_url":    f"data:image/png;base64,{enhanced_b64}",
        "psnr": p, "ssim": s,
        "model": f"zerodce_{category}",
        "brightness_class": category,
        "processing_time": elapsed,
    }

@app.post("/api/enhance/swinllie")
async def api_enhance_swinllie(image: UploadFile = File(...)):
    allowed = {"image/jpeg", "image/jpg", "image/png"}
    if image.content_type not in allowed:
        raise HTTPException(400, "Only JPG / PNG allowed.")
    contents = await image.read()
    if len(contents) > 10 * 1024 * 1024:
        raise HTTPException(400, "File too large (max 10 MB).")

    session_id = str(uuid.uuid4())
    SESSIONS[session_id] = contents

    try:
        enhanced_bytes, p, s, elapsed = enhance_quality(contents)
    except Exception as e:
        raise HTTPException(500, str(e))

    orig_b64     = base64.b64encode(contents).decode()
    enhanced_b64 = base64.b64encode(enhanced_bytes).decode()

    return {
        "success": True,
        "session_id": session_id,
        "original_url": f"data:image/png;base64,{orig_b64}",
        "image_url":    f"data:image/png;base64,{enhanced_b64}",
        "psnr": p, "ssim": s,
        "model": "swinllie",
        "processing_time": elapsed,
    }

@app.post("/api/enhance/diffunet/upload")
async def api_enhance_diffunet_upload(image: UploadFile = File(...)):
    """Direct DiffuNet upload (no prior /fast call needed)."""
    allowed = {"image/jpeg", "image/jpg", "image/png"}
    if image.content_type not in allowed:
        raise HTTPException(400, "Only JPG / PNG allowed.")
    contents = await image.read()
    if len(contents) > 10 * 1024 * 1024:
        raise HTTPException(400, "File too large (max 10 MB).")

    session_id = str(uuid.uuid4())
    SESSIONS[session_id] = contents

    try:
        enhanced_bytes, p, s, elapsed = enhance_diffunet(contents)
    except Exception as e:
        import traceback; traceback.print_exc()
        raise HTTPException(500, str(e))

    orig_b64     = base64.b64encode(contents).decode()
    enhanced_b64 = base64.b64encode(enhanced_bytes).decode()

    return {
        "success": True,
        "session_id": session_id,
        "original_url": f"data:image/png;base64,{orig_b64}",
        "image_url":    f"data:image/png;base64,{enhanced_b64}",
        "psnr": p, "ssim": s,
        "model": "diffunet",
        "processing_time": elapsed,
    }


print("✅ FastAPI app defined (fast, quality, diffunet, direct endpoints)")

✅ FastAPI app defined (fast, quality, diffunet, direct endpoints)


In [9]:
#added due to ngrok update error 

import os
import nest_asyncio
import uvicorn
from pyngrok import ngrok, conf

nest_asyncio.apply()

# --- FIX FOR 403 FORBIDDEN ERROR ---
# 1. Download ngrok manually using curl to bypass the urllib block
print("Downloading ngrok manually...")
os.system("curl -s -O https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz")

# 2. Extract the downloaded archive
os.system("tar -xf ngrok-v3-stable-linux-amd64.tgz")

# 3. Tell pyngrok to use this manually downloaded executable
conf.get_default().ngrok_path = os.path.abspath("./ngrok")
# -----------------------------------


In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()
#previous token
#NGROK_AUTH_TOKEN = "3Bj9jTzYamNKMWFbEWeCyFg5Ze8_4qg3QnYF9pCv9ZfyoQQ6K"

#new token
NGROK_AUTH_TOKEN = "3ApZyvKPKsGsdrx92RbXIbLcIZ7_7n6XyPqv8SbTc7xkseoeb"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()
public_url = ngrok.connect(8000)

print("=" * 60)
print(f"  🚀 Backend URL: {public_url}")
print("  Copy this URL into .env.local as NEXT_PUBLIC_BACKEND_URL")
print("=" * 60)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [57]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


  🚀 Backend URL: NgrokTunnel: "https://gleamingly-limitary-georgie.ngrok-free.dev" -> "http://localhost:8000"
  Copy this URL into .env.local as NEXT_PUBLIC_BACKEND_URL
🔍 Classified as: very_dark
INFO:     2402:4000:b250:a41d:758b:6a5e:fbb6:c277:0 - "POST /api/enhance/fast HTTP/1.1" 200 OK
INFO:     2402:4000:b250:a41d:758b:6a5e:fbb6:c277:0 - "OPTIONS /api/enhance/quality HTTP/1.1" 200 OK
INFO:     2402:4000:b250:a41d:758b:6a5e:fbb6:c277:0 - "OPTIONS /api/enhance/diffunet HTTP/1.1" 200 OK
INFO:     2402:4000:b250:a41d:758b:6a5e:fbb6:c277:0 - "POST /api/enhance/quality HTTP/1.1" 200 OK
INFO:     2402:4000:b250:a41d:758b:6a5e:fbb6:c277:0 - "POST /api/enhance/diffunet HTTP/1.1" 200 OK
🔍 Classified as: very_dark
INFO:     2402:4000:b250:a41d:758b:6a5e:fbb6:c277:0 - "POST /api/enhance/fast HTTP/1.1" 200 OK
INFO:     2402:4000:b250:a41d:758b:6a5e:fbb6:c277:0 - "OPTIONS /api/enhance/quality HTTP/1.1" 200 OK
INFO:     2402:4000:b250:a41d:758b:6a5e:fbb6:c277:0 - "OPTIONS /api/enhance/diffunet H